# 9.2 Specifying Data Using Python Dictionaries

Python dictionaries are the most natural data structure for supplying indexed parameter data to AMPL. A dictionary maps keys to values, and an AMPL indexed parameter maps index elements to numerical values: the correspondence is direct. For a one-dimensional parameter such as `cost {FOOD}`, the dictionary keys are food items and the values are their costs. For a two-dimensional parameter such as `amt {NUTR, FOOD}`, the keys are `(nutrient, food)` tuples and the values are the corresponding nutrient amounts. In both cases, the dictionary can be passed directly to `ampl.param`, and amplpy handles the mapping automatically.

This section illustrates the full range of dictionary-based data assignment using the diet model.

In [1]:
# install dependencies
%pip install -q amplpy pandas numpy polars

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Note: you may need to restart the kernel to use updated packages.
Licensed to AMPL Community Edition License for <uykun96@gmail.com>.


We begin by writing the diet model to a file and loading it into AMPL. The model declares two sets, `FOOD` and `NUTR`, and five parameters: the unit cost and consumption bounds for each food item, the nutritional bounds for each nutrient, and the nutrient content per serving of each food.

In [2]:
%%writefile diet.mod

set NUTR;
set FOOD;
set LINKS within (NUTR cross FOOD);

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];
param amt {NUTR,FOOD} >= 0;

Overwriting diet.mod


In [3]:
ampl.read("diet.mod")

## 9.2.1 One-Dimensional Parameters

The simplest case is a parameter indexed over a single set. Consider the parameter `cost {FOOD}`, which records the unit price of each food item. A Python dictionary maps each food name to its cost, and both the set and the parameter can be loaded from this single data structure:

In [4]:
# Define the cost parameter for each food item
cost = {
    "BEEF": 3.59,  # Cost per unit of BEEF
    "CHK":  2.59,  # -//- CHK
    "FISH": 2.29,  # -//- FISH
    "HAM":  2.89,  # -//- HAM
    "MCH":  1.89,  # -//- MCH
    "MTL":  1.99,  # -//- MTL
    "SPG":  1.99,  # -//- SPG
    "TUR":  2.49}  # -//- TUR

The set and parameter are loaded in two lines:

In [5]:
ampl.set["FOOD"] = cost.keys()   # Assign elements to the FOOD set
ampl.param["cost"] = cost        # Assign values to the cost parameter

Assigning `cost.keys()` to `ampl.set["FOOD"]` populates the AMPL set with the eight food names. Assigning the dictionary itself to `ampl.param["cost"]` then sets the value of `cost[j]` for each food `j`. Because the keys of the dictionary already name the members of `FOOD`, the two assignments are guaranteed to be consistent.

The same pattern applies to the consumption-bound parameters `f_min` and `f_max`, each of which is also indexed over `FOOD`:

In [ ]:
f_min = {"BEEF": 2, "CHK": 2, "FISH": 2, "HAM": 2, "MCH": 2, "MTL": 2, "SPG": 2, "TUR": 2}
f_max = {"BEEF": 10, "CHK": 10, "FISH": 10, "HAM": 10, "MCH": 10, "MTL": 10, "SPG": 10, "TUR": 10}

ampl.param["f_min"] = f_min
ampl.param["f_max"] = f_max

The nutrient bounds `n_min` and `n_max` follow the same pattern, but they are indexed over the set `NUTR`, which must also be populated:

In [ ]:
n_min = {"A": 700,   "C": 700,   "B1": 700,   "B2": 700,   "NA": 0,     "CAL": 16000}
n_max = {"A": 20000, "C": 20000, "B1": 20000, "B2": 20000, "NA": 50000, "CAL": 24000}

ampl.set["NUTR"] = list(n_min.keys())
ampl.param["n_min"] = n_min
ampl.param["n_max"] = n_max

## 9.2.2 Two-Dimensional Parameters

The diet model also contains a two-dimensional parameter, `amt {NUTR, FOOD}`, which records the amount of each nutrient provided by each food item. A parameter indexed over two sets is best represented in Python by a dictionary whose keys are *tuples*. Each tuple `(i, j)` identifies a `(nutrient, food)` pair, and the corresponding dictionary value is the nutrient content for that combination.

For the diet problem, the `amt` data is sparse in general: not every food provides every nutrient. The AMPL model handles this by restricting valid index pairs to the set `LINKS`, which contains only the `(nutrient, food)` pairs for which a non-zero nutrient amount is defined. Both `LINKS` and the `amt` parameter can be populated from the same dictionary:

In [ ]:
amt = {
    ("A",   "BEEF"): 60,  ("C",   "BEEF"): 20,  ("B1",  "BEEF"): 10,  ("B2",  "BEEF"): 15,
    ("NA",  "BEEF"): 928, ("CAL", "BEEF"): 295,
    ("A",   "CHK"):  8,   ("B1",  "CHK"):  20,   ("B2",  "CHK"):  20,  ("NA",  "CHK"):  2180,
    ("CAL", "CHK"):  770,
    ("A",   "FISH"): 8,   ("C",   "FISH"): 10,   ("B1",  "FISH"): 15,  ("B2",  "FISH"): 10,
    ("NA",  "FISH"): 945, ("CAL", "FISH"): 440,
    ("A",   "HAM"):  40,  ("C",   "HAM"):  40,   ("B1",  "HAM"):  35,  ("B2",  "HAM"):  10,
    ("NA",  "HAM"):  278, ("CAL", "HAM"):  430,
    ("A",   "MCH"):  15,  ("C",   "MCH"):  35,   ("B1",  "MCH"):  15,  ("B2",  "MCH"):  15,
    ("NA",  "MCH"):  1182,("CAL", "MCH"):  315,
    ("A",   "MTL"):  70,  ("C",   "MTL"):  30,   ("B1",  "MTL"):  15,  ("B2",  "MTL"):  15,
    ("NA",  "MTL"):  896, ("CAL", "MTL"):  400,
    ("A",   "SPG"):  25,  ("C",   "SPG"):  50,   ("B1",  "SPG"):  25,  ("B2",  "SPG"):  15,
    ("NA",  "SPG"):  1329,("CAL", "SPG"):  379,
    ("A",   "TUR"):  60,  ("C",   "TUR"):  20,   ("B1",  "TUR"):  15,  ("B2",  "TUR"):  10,
    ("NA",  "TUR"):  1397,("CAL", "TUR"):  450,
}

ampl.set["LINKS"] = set(amt.keys())
ampl.param["amt"] = amt

The `LINKS` set is populated from the dictionary keys by calling `set(amt.keys())`, which collects all `(nutrient, food)` pairs into a Python set before passing them to AMPL. The `amt` parameter is then assigned the entire dictionary at once. amplpy reads each `(key, value)` pair and sets `amt[nutrient, food]` to the corresponding amount.

This tuple-key pattern scales to parameters indexed over more than two sets by extending the tuples to the appropriate length. For a parameter indexed over three sets, each key would be a three-element tuple `(i, j, k)`, and the assignment syntax remains unchanged.